# Validation: Hype vs Excitement

Compare predicted hype scores against actual game excitement to validate the algorithm.

Current rating snapshots are not historical pregame evidence. The scoring functions leave qualification unavailable for past or started games. Do not interpret these comparisons as validation of historical playoff stakes, or use them to tune those stakes. Strength and championship inputs also lack a historical evidence guarantee. The evaluation time is explicit, and team lookups include the game season.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from hype_score.database import get_engine, load_games, load_team_ratings_for_games, load_pregame_odds, load_odds_snapshots
from hype_score.config import HypeConfig
from hype_score.scoring import calculate_hype_scores_batch
from hype_score.excitement import calculate_excitement_batch
from hype_score.validation import (
    compare_hype_vs_excitement,
    identify_mispredictions,
    calculate_metrics_by_sport,
    calculate_metrics_by_hype_level
)

## Load and Calculate Scores

In [ ]:
evaluated_at = pd.Timestamp.now(tz="UTC")
engine = get_engine()
config = HypeConfig.default()

games = load_games(engine, status='final', limit=1000)
pregame_odds = load_pregame_odds(engine, game_ids=games['id'].tolist())
ratings = load_team_ratings_for_games(engine, games, evaluated_at=evaluated_at)

print(f"Loaded {len(games):,} completed games")

In [ ]:
hype_results = calculate_hype_scores_batch(games, pregame_odds, ratings, config, evaluated_at=evaluated_at)
print(f"Calculated hype scores for {len(hype_results):,} games")

In [ ]:
all_snapshots = load_odds_snapshots(engine, game_ids=hype_results['game_id'].tolist())
excitement_results = calculate_excitement_batch(all_snapshots)
print(f"Calculated excitement for {len(excitement_results):,} games")

## Compare Predictions

In [ ]:
comparison = compare_hype_vs_excitement(hype_results, excitement_results)

print("Overall Validation Metrics:")
print(f"  Correlation: {comparison['correlation']:.3f}" if comparison['correlation'] else "  Correlation: N/A")
print(f"  P-value: {comparison['p_value']:.4f}" if comparison['p_value'] else "  P-value: N/A")
print(f"  MAE: {comparison['mae']:.2f}" if comparison['mae'] else "  MAE: N/A")
print(f"  RMSE: {comparison['rmse']:.2f}" if comparison['rmse'] else "  RMSE: N/A")
print(f"  Classification Accuracy: {comparison.get('classification_accuracy', 0)*100:.1f}%")
print(f"  Games compared: {comparison['n_games']:,}")

In [ ]:
merged = comparison['merged_data']

fig = px.scatter(
    merged,
    x='hype_score',
    y='volatility_score',
    color='sport',
    title='Predicted Hype vs Actual Excitement',
    labels={'hype_score': 'Hype Score (Predicted)', 'volatility_score': 'Excitement Score (Actual)'},
    hover_data=['game_id']
)

fig.add_trace(go.Scatter(
    x=[0, 100], y=[0, 100],
    mode='lines',
    name='Perfect Prediction',
    line=dict(dash='dash', color='gray')
))

fig.update_layout(xaxis_range=[0, 100], yaxis_range=[0, 100])
fig.show()

## Metrics by Sport

In [ ]:
sport_metrics = calculate_metrics_by_sport(merged)
sport_metrics

In [ ]:
fig = px.bar(
    sport_metrics,
    x='sport',
    y='correlation',
    color='classification_accuracy',
    title='Prediction Correlation by Sport',
    color_continuous_scale='RdYlGn'
)
fig.show()

## Metrics by Hype Level

In [ ]:
level_metrics = calculate_metrics_by_hype_level(merged)
level_metrics

In [ ]:
fig = px.bar(
    level_metrics,
    x='hype_level',
    y='mean_error',
    color='mean_error',
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    title='Mean Prediction Error by Hype Level<br>(Positive = Over-hyped, Negative = Under-hyped)'
)
fig.show()

## Biggest Mispredictions

In [ ]:
over_hyped, under_hyped = identify_mispredictions(merged, threshold=25)

In [ ]:
print(f"Over-hyped games ({len(over_hyped)}):")
over_hyped_display = over_hyped.merge(games[['id', 'home_team', 'away_team']], left_on='game_id', right_on='id')
over_hyped_display[['home_team', 'away_team', 'sport', 'hype_score', 'volatility_score', 'prediction_error']].head(10)

In [ ]:
print(f"Under-hyped games ({len(under_hyped)}):")
under_hyped_display = under_hyped.merge(games[['id', 'home_team', 'away_team']], left_on='game_id', right_on='id')
under_hyped_display[['home_team', 'away_team', 'sport', 'hype_score', 'volatility_score', 'prediction_error']].head(10)

## Error Distribution

In [ ]:
fig = px.histogram(
    merged,
    x='prediction_error',
    color='sport',
    nbins=50,
    title='Prediction Error Distribution (Hype - Excitement)'
)
fig.add_vline(x=0, line_dash='dash', annotation_text='Perfect')
fig.show()

## Classification Confusion Matrix

In [ ]:
confusion = pd.crosstab(
    merged['hype_level'], 
    merged['excitement_level'],
    margins=True
)
confusion

In [ ]:
conf_matrix = pd.crosstab(merged['hype_level'], merged['excitement_level'])
order = ['low', 'medium', 'high']
conf_matrix = conf_matrix.reindex(index=order, columns=order, fill_value=0)

fig = px.imshow(
    conf_matrix,
    labels=dict(x='Actual Excitement Level', y='Predicted Hype Level', color='Count'),
    title='Classification Confusion Matrix',
    text_auto=True
)
fig.show()